# Womens Soccer Project — Colab Backend Runner
Run this notebook in Google Colab to start the Flask backend on Colab GPU and expose it through an ngrok URL.

**Runtime:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1) Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')

In [ ]:
# 2) Clone the repo and switch to the correct branch
!rm -rf womens-soccer-proj
!git clone https://github.com/samayasankuratri/womens-soccer-proj.git
%cd womens-soccer-proj
!git switch frontend-fixing-samaya
!git pull origin frontend-fixing-samaya

In [ ]:
# 3) Install backend dependencies
!pip install -q flask flask-cors numpy pandas opencv-python matplotlib supervision ultralytics tqdm scipy pillow umap-learn transformers sentencepiece protobuf pyngrok

In [ ]:
# 4) Confirm important packages import correctly
import flask, numpy, cv2, supervision, ultralytics, torch
print('Imports OK')
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 5) Patch app.py to use sys.executable (correct Python) and auto-detect GPU
from pathlib import Path
import sys

app_path = Path('backend/app.py')
text = app_path.read_text()

# Ensure sys is imported
if 'import sys' not in text:
    text = text.replace('import subprocess', 'import subprocess\nimport sys', 1)

# Replace bare 'python' subprocess call with sys.executable
text = text.replace('["python", "main.py"', '[sys.executable, "main.py"')
text = text.replace("['python', 'main.py'", "[sys.executable, 'main.py'")
text = text.replace('["python", "analyze.py"', '[sys.executable, "main.py"')
text = text.replace("['python', 'analyze.py'", "[sys.executable, 'main.py'")

# Replace hardcoded cuda/cpu device with auto-detection
if '_detect_device' not in text:
    device_fn = '''
def _detect_device():
    try:
        import torch
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError:
        return 'cpu'
'''
    text = text.replace('app = Flask(__name__)', device_fn + 'app = Flask(__name__)', 1)
    text = text.replace("'--device', 'cuda'", "'--device', _detect_device()")
    text = text.replace("'--device', 'cpu'", "'--device', _detect_device()")
    text = text.replace('"--device", "cuda"', '"--device", _detect_device()')
    text = text.replace('"--device", "cpu"', '"--device", _detect_device()')

app_path.write_text(text)
print('Patched backend/app.py')
!grep -n 'sys.executable\|_detect_device\|device' backend/app.py

In [ ]:
# 6) Set your ngrok auth token
# Get/rotate yours at https://dashboard.ngrok.com → Your Authtoken
# WARNING: never commit this token to GitHub
NGROK_AUTH_TOKEN = 'PASTE_YOUR_NGROK_AUTH_TOKEN_HERE'

In [ ]:
# 7) Kill any existing process on port 5000
import time
!fuser -k 5000/tcp 2>/dev/null || true
time.sleep(2)
print('Port 5000 is clear')

In [ ]:
# 8) Start Flask backend and ngrok tunnel
from pyngrok import ngrok
import subprocess, sys, time, socket

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

process = subprocess.Popen(
    [sys.executable, 'backend/app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**__import__('os').environ, 'FLASK_DEBUG': '0'},
)

# Wait until Flask is actually listening (up to 30s)
print('Waiting for Flask to start...', end='')
for _ in range(60):
    try:
        with socket.create_connection(('localhost', 5000), timeout=1):
            print(' ready!')
            break
    except OSError:
        print('.', end='', flush=True)
        time.sleep(0.5)
else:
    print('\n⚠️  Flask did not start. Run cell 9 to see errors.')

tunnel = ngrok.connect(5000, domain='canyon-pretext-robbing.ngrok-free.dev')
public_url = tunnel.public_url
print('\n✅ Backend live at:', public_url)
print('✅ /analyze endpoint:', public_url + '/analyze')
print('Frontend .env.local should have:')
print(f'   REACT_APP_API_URL={public_url}')

In [ ]:
# 9) Watch backend logs
# Keep this cell running while you upload from the frontend.
# You should see POST /analyze when the frontend sends a video.
# Stop this cell (■) when you are done.
for line in process.stdout:
    print(line, end='')

## Frontend

On your laptop, make sure `soccer-frontend/.env.local` contains:

```
REACT_APP_API_URL=https://canyon-pretext-robbing.ngrok-free.dev
```

Then start the frontend:

```bash
cd ~/womens-soccer-proj/soccer-frontend
npm install
npm start
```

If Colab disconnects, re-run cells 7 → 8 to restart Flask and ngrok.